# Transcribe audio

In [ ]:
#@title Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Transcribe
file_path = "" #@param {type:"string"}
language = "auto" #@param ["auto", "fa", "en", "ar", "tr", "de", "fr", "es"]
model_size = "large-v3" #@param ["large-v3", "medium", "small"]

import os
if not file_path:
    raise SystemExit("Enter a file path.")
if not os.path.isfile(file_path):
    raise SystemExit(f"Not a file: {file_path}")
# faster-whisper 1.2.1 passes metadata_errors to av.open(), which av 19 removed (SYSTRAN/faster-whisper#1589).
!pip install -q faster-whisper "av<19"
import ctranslate2
from faster_whisper import WhisperModel
if ctranslate2.get_cuda_device_count():
    model = WhisperModel(model_size, device="cuda", compute_type="float16")
else:
    print("No GPU: Runtime > Change runtime type > T4 GPU. Running on CPU (slow).")
    model = WhisperModel(model_size, device="cpu", compute_type="int8")
segments, info = model.transcribe(file_path, language=None if language == "auto" else language, vad_filter=True)
print("Language:", info.language)

def ts(t):
    ms = round(t * 1000)
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

base = os.path.splitext(file_path)[0]
with open(f"{base}.srt", "w", encoding="utf-8") as srt, open(f"{base}.txt", "w", encoding="utf-8") as txt:
    for i, seg in enumerate(segments, 1):
        text = seg.text.strip()
        srt.write(f"{i}\n{ts(seg.start)} --> {ts(seg.end)}\n{text}\n\n")
        txt.write(text + "\n")
        print(f"[{ts(seg.start)[:8]}] {text}")
print("✅", f"{base}.srt", f"{base}.txt")
